<a href="https://colab.research.google.com/github/ayushiyadav02/vertex-ai-scientific-rag-benchmark/blob/main/vertex_ai_scientific_rag_benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔬 Production Scientific RAG Pipeline with Vertex AI & Gemini 2.5 Flash

This notebook demonstrates a Retrieval Augmented Generation (RAG) pipeline designed to extract and summarize information from a scientific research paper (arXiv:2202.04944) using Google Cloud's Vertex AI and Gemini 2.5 Flash.

### Architecture Overview:

1.  **Document Ingestion:** A PDF research paper is downloaded, its text extracted, and then divided into manageable `chunks`.
2.  **Embedding Generation:** Each text `chunk` is converted into a high-dimensional vector `embedding` using a Vertex AI text embedding model.
3.  **Vector Database:** These `embeddings` are stored in a `ChromaDB` instance, enabling efficient semantic search.
4.  **Retrieval Augmented Generation (RAG):** When a user asks a question, a similar embedding is generated for the `query`. The vector database is queried for the most relevant text `chunks`, and these `chunks` are then provided as `context` to the `Gemini 2.5 Flash` model to generate a grounded, factual answer.

### Tech Stack:

*   **PDF Processing:** `pypdf` for text extraction, `langchain-text-splitters` for chunking.
*   **Embeddings:** Vertex AI `text-embedding-004` (768-dimensional).
*   **Vector Database:** `ChromaDB` (in-memory for this demo).
*   **Large Language Model (LLM):** Vertex AI `Gemini 2.5 Flash` for generation.

---

### Step 1: Install Dependencies

In [ ]:
!pip install --upgrade google-cloud-aiplatform chromadb pypdf langchain-text-splitters --quiet

### Step 2: Google Cloud Authentication & Vertex AI Initialization

This section handles authenticating your Google Colab environment to access Google Cloud services and initializes the Vertex AI SDK. This is crucial for interacting with Vertex AI's embedding and generative models.

In [ ]:
from google.colab import auth
from google.auth import default
import vertexai

# 1. Authenticate the user
auth.authenticate_user()

# 2. Get credentials and initialize
credentials, project_id = default()
vertexai.init(project="YOUR_GCP_PROJECT_ID", location="us-central1", credentials=credentials)

print("Gemini API initialized successfully!")

Gemini API initialized successfully!


### Step 3: Fetch Scientific Research Paper & Chunk Text

Here, we download a specific scientific paper from arXiv (arXiv:2202.04944.pdf), extract its textual content, and then process it into smaller, manageable `chunks` using a `RecursiveCharacterTextSplitter`. Each chunk is configured to have a `chunk_size` of 500 characters with a `chunk_overlap` of 50 characters to maintain context across splits.

### Enable Vertex AI API

> **Prerequisites:** Make sure the **Vertex AI API** (`aiplatform.googleapis.com`) is enabled on your Google Cloud project before running this notebook.

In [ ]:
import urllib.request
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pypdf

# Download a public sample PDF
# Using a more reliably accessible public PDF from Google's research site
pdf_url = "https://arxiv.org/pdf/2202.04944.pdf" # A research paper from arXiv
pdf_path = "sample_report.pdf"
urllib.request.urlretrieve(pdf_url, pdf_path)

# Extract text from PDF
reader = pypdf.PdfReader(pdf_path)
raw_text = ""
for page in reader.pages:
    raw_text += page.extract_text()

# Split text into manageable chunks (chunk size ~500 chars)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_text(raw_text)

print(f"Extracted {len(reader.pages)} pages and created {len(chunks)} text chunks.")

Extracted 37 pages and created 252 text chunks.


### Step 4: Generate 768-Dimensional Vector Embeddings

This step leverages Vertex AI's `text-embedding-004` model to convert each text `chunk` into a 768-dimensional vector embedding. Batch processing is implemented for efficiency in generating these embeddings.

In [ ]:
from vertexai.language_models import TextEmbeddingInput, TextEmbeddingModel

# Load the Vertex AI Text Embedding Model
embedding_model = TextEmbeddingModel.from_pretrained("text-embedding-004")

def get_vertex_embeddings(texts, batch_size=50):
    """Generate vector embeddings in batches using Vertex AI."""
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        inputs = [TextEmbeddingInput(text=t, task_type="RETRIEVAL_DOCUMENT") for t in batch]
        kwargs = {"output_dimensionality": 768}

        embeddings = embedding_model.get_embeddings(inputs, **kwargs)
        all_embeddings.extend([e.values for e in embeddings])

    return all_embeddings

print("Generating embeddings for all 252 chunks...")
chunk_vectors = get_vertex_embeddings(chunks)
print(f"✅ Successfully generated {len(chunk_vectors)} vectors!")

/usr/local/lib/python3.12/dist-packages/vertexai/_model_garden/_model_garden_models.py:278: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


Generating embeddings for all 252 chunks...
✅ Successfully generated 252 vectors!


### Step 5: Index Embeddings in ChromaDB

After generating the embeddings, they are stored in an in-memory `ChromaDB` instance. This vector database allows for fast and efficient retrieval of relevant document `chunks` based on semantic similarity to a query.

In [ ]:
import chromadb

# Initialize ChromaDB in-memory client
chroma_client = chromadb.Client()

# Create a collection for your arXiv paper, or get it if it already exists
collection = chroma_client.get_or_create_collection(name="arxiv_rag_demo")

# Store all chunks and vectors into ChromaDB
print("Indexing chunks into ChromaDB...")
collection.add(
    documents=chunks,
    embeddings=chunk_vectors,
    ids=[f"doc_{idx}" for idx in range(len(chunks))]
)

print(f"✅ Indexed {collection.count()} items in ChromaDB!")

Indexing chunks into ChromaDB...
✅ Indexed 252 items in ChromaDB!


### Step 6: Semantic Vector Retrieval & Grounded Generation with Gemini 2.5 Flash

This final step demonstrates the core RAG functionality:

1.  **Query Embedding:** The `user_query` is transformed into a vector embedding using the same Vertex AI embedding model.
2.  **Top-K Retrieval:** `ChromaDB` is queried to retrieve the top 12 most semantically similar text `chunks` (k=12) to the user's query. The `retrieval_latency_ms` tracks the time taken for this step.
3.  **Grounded Generation:** The retrieved `chunks` are then provided as `context` to the `Gemini 2.5 Flash` model. This ensures that the model generates an answer that is 'grounded' in the provided source material, reducing the likelihood of hallucinations.

In [ ]:
import time
from vertexai.generative_models import GenerativeModel
from vertexai.language_models import TextEmbeddingInput

# Query targeting the methodology & model sections of the paper
user_query = "What supervised learning models, neural network architectures, and algorithms such as CNN, LSTM, MLP, or trees are detailed in the methodology?"

# --- STEP 1: Vector Retrieval via ChromaDB ---
start_time = time.time()

# 1. Generate 768-dim query embedding using Vertex AI
query_input = TextEmbeddingInput(text=user_query, task_type="RETRIEVAL_QUERY")
query_vector = embedding_model.get_embeddings([query_input], output_dimensionality=768)[0].values

# 2. Query top 12 chunks from ChromaDB
results = collection.query(
    query_embeddings=[query_vector],
    n_results=12
)

retrieval_latency_ms = (time.time() - start_time) * 1000
retrieved_context = "\n---\n".join(results['documents'][0])

print(f"⏱️ Vector Retrieval Latency: {retrieval_latency_ms:.2f} ms")

# --- STEP 2: Grounded Generation with Gemini 2.5 Flash ---
gemini_model = GenerativeModel("gemini-2.5-flash")

prompt = f"""
You are an expert scientific researcher. Review the context below and summarize the specific machine learning architectures, models, and methods evaluated in the paper.

Context:
{retrieved_context}

Question: {user_query}
"""

gen_start_time = time.time()
response = gemini_model.generate_content(prompt)
gen_latency_ms = (time.time() - gen_start_time) * 1000

print(f"⏱️ Gemini Generation Latency: {gen_latency_ms:.2f} ms\n")
print("--- GEMINI RESPONSE ---")
if response.candidates and response.candidates[0].content.parts:
    print(response.text)
else:
    print("⚠️ Response empty or blocked:", response.candidates[0].finish_reason)

⏱️ Vector Retrieval Latency: 140.62 ms


/usr/local/lib/python3.12/dist-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


⏱️ Gemini Generation Latency: 9974.84 ms

--- GEMINI RESPONSE ---
The study evaluates four specific supervised learning algorithms:

1.  **Multilayer Perceptron (MLP)**
    *   **Architecture:** Consists of one or more dense (fully connected) layers followed by one dense output layer.
    *   **Hyperparameters (search domain):** Up to 10 layers, with up to 200 neurons per layer.
    *   **Implementation:** Implemented using TensorFlow.

2.  **Regression Tree (RT)**
    *   **Architecture:** One tree is used per target (LLE - Lyapunov Exponent Estimator, implied).
    *   **Hyperparameters (search domain):**
        *   `maximum depth`: [1, 100]
        *   `maximum leaf nodes`: [5, 100]
        *   `maximum features`: {None, log2, square root}
        *   `splitter`: {best, random}
    *   **Reference:** Breiman et al., 1984.

3.  **Convolutional Neural Network (CNN)**
    *   **Architecture:** Comprises one 1D-convolution layer, a max pooling layer, a flatten layer, and one or more de